# 🤖 Impacto da IA em áreas de TI: análise exploratória

**Autor:** Lucas Negri

## Objetivo
Entender como profissionais de diferentes áreas de TI percebem o impacto da Inteligência Artificial no trabalho (positivo ou negativo), e como isso se relaciona com experiência, uso das ferramentas e preocupação com o emprego.

## Sobre os dados
> ⚠️ O arquivo `dados/ia_impacto_ti.csv` é um **conjunto de dados simulado**, criado para fins de estudo. Ele **não representa uma pesquisa real**, e as conclusões valem apenas para este exercício.

## Perguntas
1. Quais áreas percebem o impacto mais positivo e quais o mais negativo?
2. A variação de produtividade muda conforme a área?
3. Em quais áreas a preocupação em perder o emprego é maior?
4. Estagiários e juniores se preocupam mais do que seniores?
5. Em quais áreas as atribuições mais mudaram?
6. Quem usa IA com mais frequência percebe um impacto melhor?
7. Fazer curso de IA está ligado a mais impacto positivo e menos preocupação?
8. Quais ferramentas são mais usadas em cada área?
9. Como uso e preocupação mudaram de 2023 a 2026?
10. Há diferença por região ou tipo de empresa?
11. Quais problemas de qualidade existem nos dados e como foram tratados?

## 1. Importar bibliotecas e carregar os dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

Path("../imagens").mkdir(exist_ok=True)

df = pd.read_csv("../dados/ia_impacto_ti.csv")
df.head()

## 2. Primeira olhada nos dados

In [ ]:
print("Linhas e colunas:", df.shape)
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
# Valores nulos por coluna
nulos = df.isna().sum()
nulos[nulos > 0]

> **Observação:** os nulos em `ferramenta_principal` são esperados, porque quem não usa IA não tem ferramenta principal. Os demais nulos são tratados na limpeza.

## 3. Limpeza e qualidade dos dados (pergunta 11)

Antes de analisar, vou procurar e tratar os problemas. Cada decisão fica registrada.

In [ ]:
# Guarda os números do 'antes' para o resumo no final
resumo = {"linhas_antes": len(df)}

# 3.1 Duplicados
resumo["duplicados"] = int(df.duplicated().sum())
print("Linhas duplicadas:", resumo["duplicados"])
df = df.drop_duplicates().reset_index(drop=True)
print("Linhas após remover duplicados:", len(df))

In [ ]:
# 3.2 Nomes de área inconsistentes
print("Antes:")
print(df["area_atuacao"].value_counts())

In [ ]:
# Padroniza: remove espaços, usa minúsculas e mapeia para o nome oficial
oficiais = {
    "desenvolvimento": "Desenvolvimento",
    "dev": "Desenvolvimento",
    "qa e testes": "QA e Testes",
    "analise de dados": "Analise de Dados",
    "suporte tecnico": "Suporte Tecnico",
    "infraestrutura e devops": "Infraestrutura e DevOps",
    "seguranca da informacao": "Seguranca da Informacao",
    "design ux/ui": "Design UX/UI",
    "gestao de projetos": "Gestao de Projetos",
}
chave = df["area_atuacao"].str.strip().str.lower()
resumo["areas_corrigidas"] = int((df["area_atuacao"] != chave.map(oficiais)).sum())
df["area_atuacao"] = chave.map(oficiais)

print("Depois:")
print(df["area_atuacao"].value_counts())
print("Áreas não reconhecidas:", df["area_atuacao"].isna().sum())

In [ ]:
# 3.3 Valores impossíveis em anos_experiencia
invalido = (df["anos_experiencia"] < 0) | (df["anos_experiencia"] > 50)
display(df.loc[invalido, ["id", "nivel_experiencia", "anos_experiencia"]])

resumo["anos_invalidos"] = int(invalido.sum())
df.loc[invalido, "anos_experiencia"] = None

In [ ]:
# 3.4 Outliers em variacao_produtividade_pct
col = "variacao_produtividade_pct"
q1, q3 = df[col].quantile([0.25, 0.75])
iqr = q3 - q1
limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Limites pelo método IQR: {limite_inf:.1f} a {limite_sup:.1f}")

df[col].plot.box(title="Variação de produtividade (%) antes do tratamento")
plt.show()

In [ ]:
# Decisão: valores fora de -60% a +120% são tratados como erro de digitação
# (esse limite é uma escolha de domínio, justifique no README) e viram nulo.
fora = (df[col] < -60) | (df[col] > 120)
resumo["outliers"] = int(fora.sum())
display(df.loc[fora, ["id", "area_atuacao", col]])
df.loc[fora, col] = None

In [ ]:
# 3.5 Nulos restantes
resumo["nulos_antes"] = int(df[["regiao", "tipo_empresa", "anos_experiencia", col]].isna().sum().sum())

# Categorias: marca como 'Nao informado' em vez de inventar um valor
df["regiao"] = df["regiao"].fillna("Nao informado")
df["tipo_empresa"] = df["tipo_empresa"].fillna("Nao informado")

# anos_experiencia: usa a mediana do mesmo nível de experiência
df["anos_experiencia"] = df["anos_experiencia"].fillna(
    df.groupby("nivel_experiencia")["anos_experiencia"].transform("median")
)

# ferramenta: quem não usa IA não tem ferramenta
df["ferramenta_principal"] = df["ferramenta_principal"].fillna("Nao usa IA")

# variação de produtividade: mantém nulo (não faz sentido inventar o valor);
# o pandas ignora nulos nas médias.
print("Nulos restantes:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# 3.6 Resumo da limpeza
resumo["linhas_depois"] = len(df)
pd.Series(resumo, name="quantidade").to_frame()

In [ ]:
# Salva a versão limpa para usar nas análises
df.to_csv("../dados/ia_impacto_ti_limpo.csv", index=False)

## 4. Análise: impacto e produtividade

### Pergunta 1: quais áreas percebem o impacto mais positivo?

In [ ]:
impacto = df.groupby("area_atuacao")["nota_impacto_ia"].mean().sort_values()

ax = impacto.plot.barh(color="#4C78A8")
ax.set_title("Nota média de impacto da IA por área (1 = negativo, 5 = positivo)")
ax.set_xlabel("Nota média")
ax.set_ylabel("")
ax.set_xlim(1, 5)
plt.tight_layout()
plt.savefig("../imagens/p1_impacto_por_area.png", dpi=150)
plt.show()
impacto.round(2).sort_values(ascending=False)

**Resposta:** _escreva aqui, com os números, o que o gráfico mostra._

### Pergunta 2: a variação de produtividade muda conforme a área?

In [ ]:
ordem = df.groupby("area_atuacao")["variacao_produtividade_pct"].median().sort_values().index
dados = [df.loc[df["area_atuacao"] == a, "variacao_produtividade_pct"].dropna() for a in ordem]

plt.boxplot(dados)
plt.xticks(range(1, len(ordem) + 1), ordem, rotation=30, ha="right")
plt.title("Variação percebida de produtividade (%) por área")
plt.ylabel("Variação (%)")
plt.tight_layout()
plt.savefig("../imagens/p2_produtividade_por_area.png", dpi=150)
plt.show()

**Resposta:** _escreva aqui._

## 5. Análise: preocupação e mudanças no trabalho

### Pergunta 3: onde a preocupação em perder o emprego é maior?

In [ ]:
preocupacao = df.groupby("area_atuacao")["preocupacao_perder_emprego"].mean().sort_values()

ax = preocupacao.plot.barh(color="#E45756")
ax.set_title("Preocupação média em perder o emprego por área (1 a 5)")
ax.set_xlabel("Nota média")
ax.set_ylabel("")
ax.set_xlim(1, 5)
plt.tight_layout()
plt.savefig("../imagens/p3_preocupacao_por_area.png", dpi=150)
plt.show()

**Resposta:** _escreva aqui._

### Pergunta 4: estagiários e juniores se preocupam mais que seniores?

In [ ]:
ordem_nivel = ["Estagio", "Junior", "Pleno", "Senior", "Gestao"]
por_nivel = df.groupby("nivel_experiencia")["preocupacao_perder_emprego"].mean().reindex(ordem_nivel)

ax = por_nivel.plot.bar(color="#F58518", rot=0)
ax.set_title("Preocupação média por nível de experiência")
ax.set_ylabel("Nota média (1 a 5)")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig("../imagens/p4_preocupacao_por_nivel.png", dpi=150)
plt.show()

**Resposta:** _escreva aqui._

### Pergunta 5: em quais áreas as atribuições mais mudaram?

In [ ]:
# Proporção de 'Sim' em cada área, em porcentagem
mudou = (df["atribuicoes_mudaram"] == "Sim").groupby(df["area_atuacao"]).mean().sort_values() * 100

# TODO: faça o gráfico de barras horizontais com 'mudou' e salve em ../imagens/p5_atribuicoes.png
mudou.round(1)

**Resposta:** _escreva aqui._

## 6. Análise: uso de IA e preparação

### Pergunta 6: quem usa IA com mais frequência percebe um impacto melhor?

In [ ]:
ordem_freq = ["Nunca", "Raramente", "Semanal", "Diario"]
por_freq = df.groupby("frequencia_uso_ia")["nota_impacto_ia"].mean().reindex(ordem_freq)

# TODO: gráfico de barras de 'por_freq' e salvar em ../imagens/p6_frequencia_impacto.png
por_freq.round(2)

**Resposta:** _escreva aqui._

### Pergunta 7: fazer curso de IA está ligado a mais impacto positivo e menos preocupação?

In [ ]:
df.groupby("fez_curso_ia")[["nota_impacto_ia", "preocupacao_perder_emprego"]].mean().round(2)

**Resposta:** _escreva aqui. Lembre que correlação não prova que o curso causou a diferença._

### Pergunta 8: quais ferramentas são mais usadas em cada área?

In [ ]:
usuarios = df[df["usa_ia_no_trabalho"] == "Sim"]
tabela = pd.crosstab(usuarios["area_atuacao"], usuarios["ferramenta_principal"], normalize="index") * 100

# TODO: gráfico de barras empilhadas (tabela.plot.barh(stacked=True)) e salvar em ../imagens/p8_ferramentas.png
tabela.round(1)

**Resposta:** _escreva aqui._

## 7. Análise: evolução e contexto

### Pergunta 9: como uso e preocupação mudaram de 2023 a 2026?

In [ ]:
por_ano = df.groupby("ano_resposta").agg(
    uso_ia=("usa_ia_no_trabalho", lambda s: (s == "Sim").mean() * 100),
    preocupacao=("preocupacao_perder_emprego", "mean"),
)

# TODO: gráfico de linhas de 'por_ano' (dois gráficos ou dois eixos) e salvar em ../imagens/p9_evolucao.png
por_ano.round(2)

**Resposta:** _escreva aqui._

### Pergunta 10: há diferença por região ou tipo de empresa?

In [ ]:
# Cuidado com grupos pequenos: veja a coluna 'respostas' antes de concluir algo.
for coluna in ["regiao", "tipo_empresa"]:
    grupo = df[df[coluna] != "Nao informado"].groupby(coluna).agg(
        respostas=("id", "count"),
        impacto=("nota_impacto_ia", "mean"),
        preocupacao=("preocupacao_perder_emprego", "mean"),
    ).round(2)
    display(grupo)

**Resposta:** _escreva aqui._

## 8. Conclusões

- **Resumo das descobertas:** _responda cada pergunta em uma frase._
- **Qualidade dos dados:** _resuma os problemas encontrados (tabela da seção 3) e como foram tratados._

## 9. Limitações

- Os dados são simulados, então os resultados não valem para o mercado real.
- Algumas respostas dependem da percepção da pessoa, que é subjetiva.
- Alguns grupos (região, tipo de empresa) têm poucas respostas.
- Mostrar uma diferença entre grupos não prova a causa dela.